# Furniture CNN 원인 분석 — 3회 통제 실험

## 실험 목적
기존 Furniture CNN에서 발생한 **Validation Accuracy / Loss의 큰 변동** 원인을 한 번에 여러 요소를 바꾸지 않고 분리해서 확인합니다.

### 고정 조건
- Framework: **PyTorch**
- Optimizer: **Adam**
- Input: **128 × 128 RGB**
- Epoch: **30**
- Seed: **42**
- 데이터 증강: **없음**
- 기본 Batch Size: **64**
- 기본 Learning Rate: **0.001**
- CNN: **Conv 32 → 64 → 128 + ReLU + MaxPool + GAP + FC128 + Dropout 0.3**

### 이번 3회 실험
| 실험 | BatchNorm | Learning Rate | Batch | 목적 |
|---|---|---:|---:|---|
| A1 Baseline | 사용 | 0.001 | 64 | 기존 이상 현상 재현 |
| A2 No BN | 제거 | 0.001 | 64 | BatchNorm 영향 분리 |
| A3 Low LR | 사용 | 0.0001 | 64 | Learning Rate 영향 분리 |

> 이번 단계에서는 **한 실험당 핵심 변수 하나만 변경**합니다.  
> Batch Size 변경은 위 3회 결과를 확인한 뒤 필요할 경우 2차 실험으로 진행합니다.


In [ ]:
# STEP 1. Google Drive 연결 + 공통 라이브러리
from google.colab import drive
drive.mount("/content/drive")

import os
import random
import time
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import ImageFile

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# STEP 2. 공통 설정
SEED = 42
IMAGE_SIZE = (128, 128)
EPOCHS = 30

# 사용자 Google Drive 경로
ZIP_PATH = "/content/drive/MyDrive/dataset/furniture.zip"

# Colab 런타임 압축 해제 경로
EXTRACT_DIR = "/content/datasets/furniture"

# 결과는 Google Drive에 저장
RESULT_ROOT = "/content/drive/MyDrive/CNN_results/Furniture_CNN_Cause_Analysis_3Experiments"
os.makedirs(RESULT_ROOT, exist_ok=True)

# 이전 실험과 동일하게 이미지 증강은 사용하지 않음
ImageFile.LOAD_TRUNCATED_IMAGES = True

print("ZIP:", ZIP_PATH)
print("결과 저장:", RESULT_ROOT)


In [ ]:
# STEP 3. 재현성 설정 함수
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print("Seed 고정:", SEED)


In [ ]:
# STEP 4. furniture.zip 압축 해제 + train/valid/test 자동 탐색
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"파일을 찾을 수 없습니다: {ZIP_PATH}\n"
        "Google Drive의 MyDrive/dataset/furniture.zip 경로를 확인하세요."
    )

# 같은 런타임에서 재실행해도 꼬이지 않도록 다시 압축 해제
shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

def find_split_dirs(search_root):
    # valid를 우선 사용하되 val / validation도 대응
    valid_aliases = ("valid", "val", "validation")

    for current, dirs, _ in os.walk(search_root):
        lower_to_real = {d.lower(): d for d in dirs}

        valid_key = next(
            (name for name in valid_aliases if name in lower_to_real),
            None
        )

        if "train" in lower_to_real and "test" in lower_to_real and valid_key:
            return {
                "root": current,
                "train": os.path.join(current, lower_to_real["train"]),
                "valid": os.path.join(current, lower_to_real[valid_key]),
                "test": os.path.join(current, lower_to_real["test"]),
            }

    raise FileNotFoundError(
        "압축을 풀었지만 train / valid / test 폴더 구조를 찾지 못했습니다."
    )

SPLITS = find_split_dirs(EXTRACT_DIR)

print("Dataset root:", SPLITS["root"])
print("Train:", SPLITS["train"])
print("Valid:", SPLITS["valid"])
print("Test :", SPLITS["test"])


## 데이터셋 확인

기존 실험 조건과 맞추기 위해:
- RGB 3채널
- 128×128 Resize
- `ToTensor()`로 0~1 범위 변환
- **데이터 증강 없음**

을 그대로 사용합니다.


In [ ]:
# STEP 5. 데이터셋 구성 및 실제 폴더 구조 검증
transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),  # 0~255 -> 0~1
])

train_dataset = datasets.ImageFolder(SPLITS["train"], transform=transform)
valid_dataset = datasets.ImageFolder(SPLITS["valid"], transform=transform)
test_dataset  = datasets.ImageFolder(SPLITS["test"], transform=transform)

# 세 split의 클래스 구성이 완전히 같은지 확인
assert train_dataset.class_to_idx == valid_dataset.class_to_idx,     "train과 valid의 클래스 폴더 구성이 다릅니다."
assert train_dataset.class_to_idx == test_dataset.class_to_idx,     "train과 test의 클래스 폴더 구성이 다릅니다."

CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print("Class 수:", NUM_CLASSES)
print("Classes:", CLASS_NAMES)
print("class_to_idx:", train_dataset.class_to_idx)
print()
print("Train images:", len(train_dataset))
print("Valid images:", len(valid_dataset))
print("Test images :", len(test_dataset))


In [ ]:
# STEP 6. 클래스별 이미지 수 확인
def class_counts(dataset):
    counts = np.bincount(dataset.targets, minlength=len(dataset.classes))
    return counts

train_counts = class_counts(train_dataset)
valid_counts = class_counts(valid_dataset)
test_counts = class_counts(test_dataset)

dataset_check = pd.DataFrame({
    "class": CLASS_NAMES,
    "train": train_counts,
    "valid": valid_counts,
    "test": test_counts,
})

display(dataset_check)

dataset_check_path = os.path.join(RESULT_ROOT, "dataset_distribution.csv")
dataset_check.to_csv(dataset_check_path, index=False, encoding="utf-8-sig")

ax = dataset_check.set_index("class")[["train", "valid", "test"]].plot(
    kind="bar",
    figsize=(10, 5)
)
ax.set_title("Furniture - Dataset Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Image Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(RESULT_ROOT, "dataset_distribution.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()


# 모델 구조

기존 PyTorch + Adam Baseline과 동일한 구조를 사용합니다.

- Conv 3→32
- Conv 32→64
- Conv 64→128
- 각 Block: `Conv → BatchNorm(실험에 따라 제거) → ReLU → MaxPool`
- Global Average Pooling
- FC 128
- Dropout 0.3
- Output = 데이터셋 클래스 수

**A2만 BatchNorm을 제거**하고 나머지 구조는 동일하게 유지합니다.


In [ ]:
# STEP 7. CNN 정의
class FurnitureCNN(nn.Module):
    def __init__(self, num_classes, use_batchnorm=True, dropout=0.3):
        super().__init__()

        def block(in_channels, out_channels):
            layers = [
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=3,
                    padding=1,
                    bias=False
                )
            ]

            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_channels))

            layers += [
                nn.ReLU(),
                nn.MaxPool2d(2)
            ]
            return nn.Sequential(*layers)

        self.block1 = block(3, 32)
        self.block2 = block(32, 64)
        self.block3 = block(64, 128)

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

# Baseline 파라미터 수 검증
_check_model = FurnitureCNN(
    num_classes=NUM_CLASSES,
    use_batchnorm=True,
    dropout=0.3
)

BASELINE_PARAMS = sum(p.numel() for p in _check_model.parameters())
print(_check_model)
print()
print("Baseline parameter count:", f"{BASELINE_PARAMS:,}")

if NUM_CLASSES == 5:
    assert BASELINE_PARAMS == 110629, (
        f"기존 5-class Baseline은 110,629개여야 하는데 "
        f"{BASELINE_PARAMS:,}개입니다."
    )
    print("✓ 기존 PyTorch Adam Baseline 110,629 parameters와 일치")


In [ ]:
# STEP 8. DataLoader 생성 함수
def make_loaders(batch_size, seed=42):
    # 실험마다 같은 seed에서 shuffle을 시작해 비교 조건을 맞춤
    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=generator,
        pin_memory=torch.cuda.is_available()
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, valid_loader, test_loader


In [ ]:
# STEP 9. 학습 / 평가 함수
def run_epoch_train(model, loader, criterion, optimizer):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return (
        total_loss / total_samples,
        total_correct / total_samples
    )


@torch.no_grad()
def run_epoch_eval(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return (
        total_loss / total_samples,
        total_correct / total_samples
    )


@torch.no_grad()
def predict_all(model, loader):
    model.eval()

    y_true = []
    y_pred = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(preds.tolist())

    return np.array(y_true), np.array(y_pred)


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
    }


In [ ]:
# STEP 10. 실험 설정
EXPERIMENTS = [
    {
        "experiment": "A1_Baseline",
        "use_batchnorm": True,
        "learning_rate": 0.001,
        "batch_size": 64,
        "dropout": 0.3,
        "purpose": "기존 Validation 불안정 현상 재현"
    },
    {
        "experiment": "A2_No_BatchNorm",
        "use_batchnorm": False,
        "learning_rate": 0.001,
        "batch_size": 64,
        "dropout": 0.3,
        "purpose": "BatchNorm이 Validation 변동에 미치는 영향 분리"
    },
    {
        "experiment": "A3_Low_LR",
        "use_batchnorm": True,
        "learning_rate": 0.0001,
        "batch_size": 64,
        "dropout": 0.3,
        "purpose": "Learning Rate가 Validation 변동에 미치는 영향 분리"
    },
]

display(pd.DataFrame(EXPERIMENTS))


In [ ]:
# STEP 11. 실험 1개를 실행하고 결과를 저장하는 함수
def run_experiment(config):
    experiment_name = config["experiment"]
    result_dir = os.path.join(RESULT_ROOT, experiment_name)
    os.makedirs(result_dir, exist_ok=True)

    print("=" * 80)
    print("실험:", experiment_name)
    print("목적:", config["purpose"])
    print("BatchNorm:", config["use_batchnorm"])
    print("LR:", config["learning_rate"])
    print("Batch:", config["batch_size"])
    print("=" * 80)

    # 각 실험을 같은 초기 조건에서 시작
    set_seed(SEED)

    train_loader, valid_loader, test_loader = make_loaders(
        batch_size=config["batch_size"],
        seed=SEED
    )

    model = FurnitureCNN(
        num_classes=NUM_CLASSES,
        use_batchnorm=config["use_batchnorm"],
        dropout=config["dropout"]
    ).to(device)

    parameter_count = sum(p.numel() for p in model.parameters())

    criterion = nn.CrossEntropyLoss()

    # 기존 실험과 동일한 Adam 세부 설정
    optimizer = optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        betas=(0.9, 0.999),
        eps=1e-7
    )

    history = []

    best_val_loss = float("inf")
    best_epoch_by_val_loss = -1
    best_model_path = os.path.join(result_dir, "best_model.pt")
    final_model_path = os.path.join(result_dir, "final_model.pt")

    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = run_epoch_train(
            model, train_loader, criterion, optimizer
        )
        val_loss, val_acc = run_epoch_eval(
            model, valid_loader, criterion
        )

        history.append({
            "epoch": epoch,
            "loss": train_loss,
            "accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch_by_val_loss = epoch
            torch.save(model.state_dict(), best_model_path)

        print(
            f"[{experiment_name}] "
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Acc {train_acc:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Acc {val_acc:.4f}"
        )

    training_seconds = time.time() - start_time

    # 마지막 epoch 모델 저장 및 평가
    torch.save(model.state_dict(), final_model_path)

    y_true_final, y_pred_final = predict_all(model, test_loader)
    final_metrics = classification_metrics(y_true_final, y_pred_final)

    # Best Validation Loss checkpoint 평가
    best_model = FurnitureCNN(
        num_classes=NUM_CLASSES,
        use_batchnorm=config["use_batchnorm"],
        dropout=config["dropout"]
    ).to(device)

    best_model.load_state_dict(
        torch.load(best_model_path, map_location=device)
    )

    y_true_best, y_pred_best = predict_all(best_model, test_loader)
    best_metrics = classification_metrics(y_true_best, y_pred_best)

    history_df = pd.DataFrame(history)

    # 원인 분석용 핵심 수치
    final_row = history_df.iloc[-1]
    best_val_acc = history_df["val_accuracy"].max()
    best_val_acc_epoch = int(
        history_df.loc[history_df["val_accuracy"].idxmax(), "epoch"]
    )

    worst_val_acc = history_df["val_accuracy"].min()
    worst_val_acc_epoch = int(
        history_df.loc[history_df["val_accuracy"].idxmin(), "epoch"]
    )

    last10 = history_df.tail(min(10, len(history_df)))

    last10_val_acc_range = (
        last10["val_accuracy"].max() - last10["val_accuracy"].min()
    )
    last10_val_acc_std = last10["val_accuracy"].std(ddof=0)

    last10_val_loss_range = (
        last10["val_loss"].max() - last10["val_loss"].min()
    )
    last10_val_loss_std = last10["val_loss"].std(ddof=0)

    summary = {
        "experiment": experiment_name,
        "purpose": config["purpose"],
        "seed": SEED,
        "epochs": EPOCHS,
        "image_size": f"{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}",
        "batch_size": config["batch_size"],
        "learning_rate": config["learning_rate"],
        "batchnorm": config["use_batchnorm"],
        "dropout": config["dropout"],
        "parameter_count": parameter_count,
        "training_seconds": training_seconds,

        "best_epoch_by_val_loss": best_epoch_by_val_loss,
        "best_val_loss": best_val_loss,
        "best_val_accuracy": best_val_acc,
        "best_val_accuracy_epoch": best_val_acc_epoch,

        "worst_val_accuracy": worst_val_acc,
        "worst_val_accuracy_epoch": worst_val_acc_epoch,

        "final_train_loss": final_row["loss"],
        "final_train_accuracy": final_row["accuracy"],
        "final_val_loss": final_row["val_loss"],
        "final_val_accuracy": final_row["val_accuracy"],
        "final_train_val_gap": (
            final_row["accuracy"] - final_row["val_accuracy"]
        ),

        "last10_val_accuracy_range": last10_val_acc_range,
        "last10_val_accuracy_std": last10_val_acc_std,
        "last10_val_loss_range": last10_val_loss_range,
        "last10_val_loss_std": last10_val_loss_std,

        "best_test_accuracy": best_metrics["accuracy"],
        "best_macro_precision": best_metrics["macro_precision"],
        "best_macro_recall": best_metrics["macro_recall"],
        "best_macro_f1": best_metrics["macro_f1"],

        "final_test_accuracy": final_metrics["accuracy"],
        "final_macro_precision": final_metrics["macro_precision"],
        "final_macro_recall": final_metrics["macro_recall"],
        "final_macro_f1": final_metrics["macro_f1"],
    }

    # CSV 저장
    history_df.to_csv(
        os.path.join(result_dir, "history.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame([summary]).to_csv(
        os.path.join(result_dir, "summary.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # Best / Final classification report 저장
    best_report = pd.DataFrame(
        classification_report(
            y_true_best,
            y_pred_best,
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )
    ).T

    final_report = pd.DataFrame(
        classification_report(
            y_true_final,
            y_pred_final,
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )
    ).T

    best_report.to_csv(
        os.path.join(result_dir, "best_classification_report.csv"),
        encoding="utf-8-sig"
    )
    final_report.to_csv(
        os.path.join(result_dir, "final_classification_report.csv"),
        encoding="utf-8-sig"
    )

    # Accuracy 그래프
    plt.figure(figsize=(9, 5))
    plt.plot(history_df["epoch"], history_df["accuracy"], label="Train Accuracy")
    plt.plot(history_df["epoch"], history_df["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{experiment_name} - Accuracy")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(
        os.path.join(result_dir, "accuracy_curve.png"),
        dpi=180,
        bbox_inches="tight"
    )
    plt.show()

    # Loss 그래프
    plt.figure(figsize=(9, 5))
    plt.plot(history_df["epoch"], history_df["loss"], label="Train Loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{experiment_name} - Loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(
        os.path.join(result_dir, "loss_curve.png"),
        dpi=180,
        bbox_inches="tight"
    )
    plt.show()

    # Best checkpoint confusion matrix
    cm = confusion_matrix(y_true_best, y_pred_best)
    fig, ax = plt.subplots(figsize=(7, 7))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_NAMES
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
    ax.set_title(f"{experiment_name} - Best Checkpoint Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        os.path.join(result_dir, "best_confusion_matrix.png"),
        dpi=180,
        bbox_inches="tight"
    )
    plt.show()

    print()
    print("핵심 결과")
    print(f"- Best Val Loss Epoch       : {best_epoch_by_val_loss}")
    print(f"- Best Val Accuracy         : {best_val_acc:.4f}")
    print(f"- Final Val Accuracy        : {final_row['val_accuracy']:.4f}")
    print(f"- Last 10 Val Acc Range     : {last10_val_acc_range:.4f}")
    print(f"- Last 10 Val Acc Std       : {last10_val_acc_std:.4f}")
    print(f"- Best Checkpoint Test Acc  : {best_metrics['accuracy']:.4f}")
    print(f"- Best Checkpoint Macro F1  : {best_metrics['macro_f1']:.4f}")
    print(f"- Final Test Acc            : {final_metrics['accuracy']:.4f}")
    print(f"- Final Macro F1            : {final_metrics['macro_f1']:.4f}")
    print(f"- Training Time             : {training_seconds:.1f} sec")
    print(f"- Saved                     : {result_dir}")

    return summary, history_df


# STEP 12. 3회 실험 실행

아래 셀 하나로 **A1 → A2 → A3**를 순서대로 실행합니다.

시간이 촉박하면:
1. A1이 끝나는 즉시 그래프를 확인
2. A2 실행
3. A3 실행
4. 마지막 비교표/그래프 셀 실행

순서로 진행하면 됩니다.


In [ ]:
all_summaries = []
all_histories = {}

for config in EXPERIMENTS:
    summary, history_df = run_experiment(config)
    all_summaries.append(summary)
    all_histories[config["experiment"]] = history_df

summary_df = pd.DataFrame(all_summaries)

summary_path = os.path.join(RESULT_ROOT, "A1_A2_A3_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\n3회 실험 완료")
print("통합 Summary:", summary_path)
display(summary_df)


# STEP 13. PPT용 핵심 비교표

가장 먼저 볼 항목은 단순 최고 정확도가 아니라 **Validation 안정성**입니다.

### 특히 비교할 값
- `last10_val_accuracy_range`: 마지막 10 Epoch의 Validation Accuracy 최대-최소 차이
- `last10_val_accuracy_std`: 마지막 10 Epoch의 Validation Accuracy 표준편차
- `best_val_accuracy` vs `final_val_accuracy`
- `best_test_accuracy`
- `best_macro_f1`

**변동폭/표준편차가 작을수록 후반 학습이 더 안정적**이라고 볼 수 있습니다.


In [ ]:
# PPT에 바로 옮기기 쉬운 핵심 지표만 출력
ppt_columns = [
    "experiment",
    "batchnorm",
    "learning_rate",
    "batch_size",
    "best_val_accuracy",
    "final_val_accuracy",
    "last10_val_accuracy_range",
    "last10_val_accuracy_std",
    "best_epoch_by_val_loss",
    "best_test_accuracy",
    "best_macro_f1",
]

ppt_table = summary_df[ppt_columns].copy()

percent_cols = [
    "best_val_accuracy",
    "final_val_accuracy",
    "last10_val_accuracy_range",
    "last10_val_accuracy_std",
    "best_test_accuracy",
    "best_macro_f1",
]

ppt_display = ppt_table.copy()
for col in percent_cols:
    ppt_display[col] = (ppt_display[col] * 100).round(2)

display(ppt_display)

ppt_table.to_csv(
    os.path.join(RESULT_ROOT, "PPT_key_metrics.csv"),
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
# STEP 14. 3개 실험 Validation Accuracy 한 그래프에서 비교
plt.figure(figsize=(10, 6))

for experiment_name, history_df in all_histories.items():
    plt.plot(
        history_df["epoch"],
        history_df["val_accuracy"],
        label=experiment_name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("A1 vs A2 vs A3 - Validation Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

comparison_acc_path = os.path.join(
    RESULT_ROOT,
    "A1_A2_A3_validation_accuracy_comparison.png"
)
plt.savefig(comparison_acc_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", comparison_acc_path)


In [ ]:
# STEP 15. 3개 실험 Validation Loss 한 그래프에서 비교
plt.figure(figsize=(10, 6))

for experiment_name, history_df in all_histories.items():
    plt.plot(
        history_df["epoch"],
        history_df["val_loss"],
        label=experiment_name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("A1 vs A2 vs A3 - Validation Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

comparison_loss_path = os.path.join(
    RESULT_ROOT,
    "A1_A2_A3_validation_loss_comparison.png"
)
plt.savefig(comparison_loss_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", comparison_loss_path)


In [ ]:
# STEP 16. 마지막 10 Epoch 안정성 비교
stability_table = summary_df[[
    "experiment",
    "last10_val_accuracy_range",
    "last10_val_accuracy_std",
    "last10_val_loss_range",
    "last10_val_loss_std",
]].copy()

stability_display = stability_table.copy()
stability_display["last10_val_accuracy_range"] *= 100
stability_display["last10_val_accuracy_std"] *= 100

display(stability_display.round(4))

# Accuracy 변동폭이 작은 순서
ranking = summary_df.sort_values(
    "last10_val_accuracy_range",
    ascending=True
)[[
    "experiment",
    "last10_val_accuracy_range",
    "last10_val_accuracy_std",
    "best_test_accuracy",
    "best_macro_f1",
]]

print("Validation Accuracy 후반 변동폭 기준 안정성 순위")
display(ranking)


# STEP 17. 결과 해석 기준

실험 결과가 나온 뒤 다음 기준으로 원인을 판단합니다.

### A2 No BatchNorm에서 변동이 크게 감소했다면
> 동일한 Learning Rate와 Batch Size 조건에서 BatchNorm 제거 후 Validation 변동이 감소했으므로, **BatchNorm의 running statistics / batch 단위 통계가 불안정성에 영향을 주었을 가능성**이 높다.

### A3 Low LR에서 변동이 크게 감소했다면
> 구조와 BatchNorm을 유지하고 Learning Rate만 낮췄을 때 Validation 변동이 감소했으므로, **기존 LR=0.001에서 parameter update 폭이 커 후반 학습이 불안정했을 가능성**이 높다.

### A2와 A3 모두 큰 개선이 없다면
> BatchNorm과 Learning Rate만으로는 현상을 충분히 설명하기 어렵다. 다음 2차 실험에서 **Batch Size**, 데이터 분포, 클래스별 오류 패턴 등을 추가 확인한다.

### 주의
- Accuracy 한 번의 최고값만 보고 결론을 내리지 않습니다.
- **후반 10 Epoch 변동폭 + 표준편차 + Best/Final 차이**를 함께 봅니다.
- 한 실험에서 한 가지 핵심 변수만 바꿔야 원인 분석의 설득력이 유지됩니다.


In [ ]:
# STEP 18. 결과 폴더 ZIP으로 묶기
archive_base = "/content/Furniture_CNN_Cause_Analysis_3Experiments"

zip_path = shutil.make_archive(
    archive_base,
    "zip",
    RESULT_ROOT
)

drive_zip_path = os.path.join(
    RESULT_ROOT,
    "Furniture_CNN_Cause_Analysis_3Experiments.zip"
)

shutil.copy2(zip_path, drive_zip_path)

print("Colab ZIP:", zip_path)
print("Drive ZIP :", drive_zip_path)
print("완료")
